## 20_ppml_distance
  PPML estimate of b with origin fixed effects and three distance specifications (spline, power,
  exponential), CZ and PL. Output: data/processed/20_ppml_distance/*.csv.

In [ ]:
# PROJ data fix: must run BEFORE geopandas / pyproj are imported
import os
import sys
from pathlib import Path
from importlib.util import find_spec


def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec and spec.submodule_search_locations:  # pip wheel with bundled data
        candidates.append(Path(list(spec.submodule_search_locations)[0]) / "proj_dir" / "share" / "proj")
    for c in candidates:
        if (c / "proj.db").exists():
            return c
    return None


proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.wkb
from geopandas.array import GeometryArray
from scipy.spatial.distance import pdist, squareform

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
YEAR = 2023                                   # population year, same as notebooks 05 and 05b
CRS_GEO, CRS_METRIC = "EPSG:4326", "EPSG:3035"
COL_ID, COL_POP = "nuts3_id", "population"
COL_COUNTRY, COL_YEAR = "country_code", "year"
NUTS3_TABLE = config.PROCESSED_DIR / "nuts3_master_table.parquet"
OUT_DIR = config.PROCESSED_DIR / "20_ppml_distance"
# country -> observed OD matrix (wide format, first column = origin id)
OD_FILES = {
    "CZ": config.PROCESSED_DIR / "05_gravity_cz" / "od_observed.csv",
    "PL": config.PROCESSED_DIR / "05b_gravity_pl" / "od_observed.csv",
}
# long-table column names (own names, English)
C_ORIGIN, C_DEST, C_FLOW = "origin", "destination", "flow"
C_DIST, C_POP_O, C_POP_D = "dist_km", "pop_origin", "pop_dest"
# ------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Master table; geometry is stored as WKB bytes
nuts3 = pd.read_parquet(NUTS3_TABLE)
geoms = nuts3["geometry"].apply(
    lambda x: shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x
).tolist()
geom_arr = np.empty(len(geoms), dtype=object)
for i, g in enumerate(geoms):
    geom_arr[i] = g
nuts3 = gpd.GeoDataFrame(nuts3.drop(columns="geometry"),
                         geometry=GeometryArray(geom_arr, crs=CRS_GEO))

summary = []
for country, od_path in OD_FILES.items():
    # Regions of the country, centroids and pairwise distances (km)
    reg = (nuts3[(nuts3[COL_COUNTRY] == country) & (nuts3[COL_YEAR] == YEAR)]
           .copy().reset_index(drop=True))
    reg_metric = reg.to_crs(CRS_METRIC)
    xy = np.array([[p.x, p.y] for p in reg_metric.geometry.centroid])
    dist = pd.DataFrame(squareform(pdist(xy)) / 1000, index=reg[COL_ID], columns=reg[COL_ID])
    pop = reg.set_index(COL_ID)[COL_POP]

    # Observed OD matrix (wide) -> long table without the diagonal
    od = pd.read_csv(od_path, index_col=0)
    assert set(od.index) == set(od.columns) <= set(pop.index), f"{country}: id mismatch"
    long = (od.rename_axis(index=C_ORIGIN, columns=C_DEST)
              .stack().rename(C_FLOW).reset_index())
    long = long[long[C_ORIGIN] != long[C_DEST]].copy()

    # Distance and masses by lookup (pairs with zero flow stay in the table)
    long[C_DIST] = dist.to_numpy()[dist.index.get_indexer(long[C_ORIGIN]),
                                   dist.columns.get_indexer(long[C_DEST])]
    long[C_POP_O] = long[C_ORIGIN].map(pop)
    long[C_POP_D] = long[C_DEST].map(pop)
    long.insert(0, COL_COUNTRY, country)

    out_path = OUT_DIR / f"od_long_{country}.csv"
    long.to_csv(out_path, index=False)
    summary.append({"country": country, "regions": len(od), "pairs": len(long),
                    "zero_share": round(float((long[C_FLOW] == 0).mean()), 3),
                    "total_flow": int(long[C_FLOW].sum()),
                    "missing_values": int(long.isna().sum().sum())})
    print("Saved:", os.path.relpath(out_path))

print(pd.DataFrame(summary).to_string(index=False))

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# ----------------------------- CONFIG (PPML) -----------------------------
SPLINE_DF = 4                     # degrees of freedom of the B-spline on log distance
SPLINE_DEGREE = 3
COV_TYPE = "HC1"                  # heteroskedasticity-robust standard errors
COL_LOG_POP_D, COL_LOG_DIST = "log_pop_dest", "log_dist"
PPML_PARAMS_CSV = OUT_DIR / "ppml_spline_params.csv"
# ------------------------------------------------------------------------

rows = []
ppml_fits = {}
for country in OD_FILES:
    long = pd.read_csv(OUT_DIR / f"od_long_{country}.csv")
    long[COL_LOG_POP_D] = np.log(long[C_POP_D])
    long[COL_LOG_DIST] = np.log(long[C_DIST])

    # Origin fixed effects + destination mass + spline of log distance
    formula = (f"{C_FLOW} ~ C({C_ORIGIN}) + {COL_LOG_POP_D}"
               f" + bs({COL_LOG_DIST}, df={SPLINE_DF}, degree={SPLINE_DEGREE})")
    fit = smf.glm(formula, data=long, family=sm.families.Poisson()).fit(cov_type=COV_TYPE)
    ppml_fits[country] = fit

    ci_low, ci_high = fit.conf_int().loc[COL_LOG_POP_D]
    rows.append({"country": country, "n_pairs": int(fit.nobs),
                 "b": fit.params[COL_LOG_POP_D], "se": fit.bse[COL_LOG_POP_D],
                 "ci_low": ci_low, "ci_high": ci_high,
                 "converged": bool(fit.converged), "cov_type": COV_TYPE,
                 "spline_df": SPLINE_DF})

ppml_params = pd.DataFrame(rows)
ppml_params.to_csv(PPML_PARAMS_CSV, index=False)
print(ppml_params.round(4).to_string(index=False))
print("Saved:", os.path.relpath(PPML_PARAMS_CSV))

In [ ]:
# ----------------------------- CONFIG (comparison) -----------------------------
SKMOB_PARAM_FILES = {
    "CZ": config.PROCESSED_DIR / "05_gravity_cz" / "gravity_params.csv",
    "PL": config.PROCESSED_DIR / "05b_gravity_pl" / "gravity_params.csv",
}
COL_B_SKMOB = "destination_exp"      # column of the skmob parameter files
B_COMPARISON_CSV = OUT_DIR / "b_comparison.csv"
# --------------------------------------------------------------------------------

rows = []
for country, path in SKMOB_PARAM_FILES.items():
    # skmob rows: one per distance-function variant
    skmob_params = pd.read_csv(path)
    n_pairs_skmob = int(pd.read_csv(OUT_DIR / f"od_long_{country}.csv")[C_FLOW].gt(0).sum())
    for _, r in skmob_params.iterrows():
        rows.append({"country": country, "method": f"skmob_{r['model']}",
                     "b": r[COL_B_SKMOB], "ci_low": np.nan, "ci_high": np.nan,
                     "n_pairs": int(r["n_fit_pairs"]) if "n_fit_pairs" in r else n_pairs_skmob})
    # PPML row from the previous cell
    p = ppml_params[ppml_params["country"] == country].iloc[0]
    rows.append({"country": country, "method": "ppml_spline", "b": p["b"],
                 "ci_low": p["ci_low"], "ci_high": p["ci_high"], "n_pairs": int(p["n_pairs"])})

b_comparison = pd.DataFrame(rows)
b_comparison.to_csv(B_COMPARISON_CSV, index=False)
print(b_comparison.round(4).to_string(index=False))
print("Saved:", os.path.relpath(B_COMPARISON_CSV))

In [ ]:
# ----------------------------- CONFIG (deterrence variants) -----------------------------
# deterrence term of the formula for each variant (spline settings come from the previous cell)
DETERRENCE_TERMS = {
    "power_law": COL_LOG_DIST,                                         # linear in log distance
    "exponential": C_DIST,                                             # linear in distance (km)
    "spline": f"bs({COL_LOG_DIST}, df={SPLINE_DF}, degree={SPLINE_DEGREE})",
}
SAMPLES = ["all_pairs", "positive_pairs"]
VARIANTS_CSV = OUT_DIR / "ppml_deterrence_variants.csv"
# ----------------------------------------------------------------------------------------

rows = []
for country in OD_FILES:
    long = pd.read_csv(OUT_DIR / f"od_long_{country}.csv")
    long[COL_LOG_POP_D] = np.log(long[C_POP_D])
    long[COL_LOG_DIST] = np.log(long[C_DIST])
    has_zeros = bool((long[C_FLOW] == 0).any())

    for sample in SAMPLES:
        # Without zero-flow pairs the two samples are identical: skip the second run
        if sample == "positive_pairs" and not has_zeros:
            continue
        data = long if sample == "all_pairs" else long[long[C_FLOW] > 0]

        for variant, term in DETERRENCE_TERMS.items():
            formula = f"{C_FLOW} ~ C({C_ORIGIN}) + {COL_LOG_POP_D} + {term}"
            fit = smf.glm(formula, data=data, family=sm.families.Poisson()).fit(cov_type=COV_TYPE)
            ci_low, ci_high = fit.conf_int().loc[COL_LOG_POP_D]
            rows.append({"country": country, "sample": sample, "deterrence": variant,
                         "n_pairs": int(fit.nobs), "b": fit.params[COL_LOG_POP_D],
                         "se": fit.bse[COL_LOG_POP_D], "ci_low": ci_low, "ci_high": ci_high,
                         "converged": bool(fit.converged)})

ppml_variants = pd.DataFrame(rows)
ppml_variants.to_csv(VARIANTS_CSV, index=False)
print(ppml_variants.round(4).to_string(index=False))
print("Saved:", os.path.relpath(VARIANTS_CSV))